# 🎛️ Timbre Transfer — Colab

Run RAVE timbre-transfer **inference** and **fine-tuning** on a free/paid GPU.

**Steps:** install → (optional) mount Drive → download a pre-trained model → run inference → launch the Gradio UI → preprocess a custom dataset → fine-tune.

> Set the runtime to GPU: **Runtime → Change runtime type → GPU**.

## 1. Get the code & install dependencies

In [ ]:
# Clone the repo (or skip if you uploaded it / are running from a checkout).
import os
REPO_URL = "https://github.com/heyuwang1999/timbre-transfer.git"
if not os.path.exists("timbre-transfer"):
    !git clone $REPO_URL
%cd timbre-transfer

In [ ]:
# Core runtime deps. Torch/torchaudio are preinstalled on Colab GPU runtimes.
!pip install -q -r requirements.txt
!pip install -q -e .
# For fine-tuning (the `rave` CLI):
!pip install -q acids-rave

## 2. (Optional) Mount Google Drive
Store models, datasets, and outputs on Drive so they persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/timbre-transfer'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive workspace:', DRIVE_ROOT)

## 3. Download a pre-trained RAVE model from Hugging Face

In [ ]:
# List what's available, then download one.
from timbre_transfer.config import load_config
from timbre_transfer.models.registry import list_available_models, resolve_entry
from timbre_transfer.utils.download import fetch_model

cfg = load_config()
for m in list_available_models(cfg):
    print(f'{m.label:12s} {m.filename}')

entry = resolve_entry('guitar', cfg)          # <-- pick any label above
model_path = fetch_model(entry.repo, entry.filename, cache_dir='models')
print('Downloaded ->', model_path)

## 4. Run inference
Upload a (monophonic) source clip, then transfer its timbre.

In [ ]:
from google.colab import files
uploaded = files.upload()           # choose a .wav/.mp3 source
src_path = next(iter(uploaded))
print('Uploaded:', src_path)

In [ ]:
import IPython.display as ipd
from timbre_transfer.config import resolve_device
from timbre_transfer.models.rave_model import load_rave
from timbre_transfer.inference.transfer import transfer_file, LatentControls

device = resolve_device('auto')
model = load_rave(model_path, device=device)
out = transfer_file(model, src_path, 'output.wav', controls=LatentControls(scale=1.0))
print('Wrote', out, '| sr =', model.sample_rate, '| device =', device)

print('Source:');  ipd.display(ipd.Audio(src_path))
print('Output:');  ipd.display(ipd.Audio('output.wav'))

## 5. Launch the interactive Gradio app
Use `share=True` to get a public link from Colab.

In [ ]:
import sys; sys.argv = ['gradio_app.py', '--share']
exec(open('app/gradio_app.py').read())

## 6. Fine-tune on your own data

⚠️ A `.ts` file is inference-only. Fine-tuning uses the `rave` CLI on a Lightning `.ckpt`.
Put your target-instrument audio in a folder (e.g. on Drive), then preprocess + train.

In [ ]:
# Preprocess into RAVE's dataset format (point --input at your audio folder).
!python scripts/preprocess_dataset.py --input ./data/my_instrument --output ./db --use-rave

In [ ]:
# Fine-tune from a pre-trained .ckpt (download one, or train from scratch by omitting --ckpt).
!python scripts/finetune.py rave --db ./db --name colab_run --config v2 --out /content/drive/MyDrive/timbre-transfer/runs

In [ ]:
# Illustrative pure-PyTorch loop (teaching scaffold; no acids-rave needed).
!python scripts/preprocess_dataset.py --input ./data/my_instrument --output ./dataset
!python scripts/finetune.py demo --manifest ./dataset/manifest.json --epochs 3 --device cuda